# SE4050 – Deep Learning 2026
## Master Model Comparison Notebook
**Task:** Multi-Class Hotel Review Sentiment Classification (3 Classes: Poor, Average, Good)

This notebook provides a fair, standardized, and comprehensive experimental comparison across **four distinct Deep Learning architectures**:
1. **Simple Recurrent Neural Network (SimpleRNN)**
2. **Long Short-Term Memory (LSTM)**
3. **Gated Recurrent Unit (GRU)**
4. **1D Convolutional Neural Network (1D-CNN)**

All models are evaluated on the exact same held-out test dataset (`X_test_pad.npy`, `y_test.npy`) using mandatory evaluation metrics:
- **Accuracy, Macro/Weighted Precision, Recall, and F1-Score**
- **Multi-Class ROC-AUC (One-vs-Rest)**
- **Confusion Matrix Heatmaps**
- **Model Complexity (Trainable Parameters) & Inference Latency**


In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Add parent directory to path for common imports
sys.path.append("..")
from common.data_loader import load_processed_data
from common.evaluation import (
    evaluate_classification_metrics,
    plot_confusion_matrix,
    plot_roc_curves,
    measure_inference_speed
)

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow Version:", tf.__version__)
print("Environment setup complete.")


---
### 1. Load Processed Test Dataset
We load the untouched, held-out test data from `data/processed/`.

In [ ]:
DATA_DIR = "../data/processed"
X_train, X_val, X_test, y_train, y_val, y_test = load_processed_data(DATA_DIR)

class_names = ["Poor", "Average", "Good"]
num_classes = len(class_names)

print(f"Training set shape:   {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape:       {X_test.shape}")
print("Test class distribution:", dict(zip(*np.unique(y_test, return_counts=True))))


---
### 2. Load Trained Deep Learning Models & Calculate Parameters
We load saved model weights from `models_saved/` and extract model complexity parameters.

In [ ]:
MODEL_DIR = "../models_saved"

model_paths = {
    "SimpleRNN": os.path.join(MODEL_DIR, "rnn_model.keras"),
    "LSTM":      os.path.join(MODEL_DIR, "lstm_model.keras"),
    "GRU":       os.path.join(MODEL_DIR, "gru_model.keras"),
    "1D-CNN":    os.path.join(MODEL_DIR, "cnn_model.keras")
}

models = {}
params = {}

for name, path in model_paths.items():
    if os.path.exists(path):
        model = tf.keras.models.load_model(path)
        models[name] = model
        params[name] = model.count_params()
        print(f"Loaded {name:<10} | Total Parameters: {params[name]:,}")
    else:
        print(f"[WARNING] Checkpoint for {name} not found at {path}")


---
### 3. Evaluate Test Performance & Measure Inference Speed
Each model generates predictions on the test set (`X_test`). We evaluate accuracy, macro/weighted metrics, multi-class ROC-AUC, and inference latency.

In [ ]:
results = []
predictions_probs = {}
predictions_classes = {}

for name, model in models.items():
    print(f"Evaluating {name}...")
    # Predict probabilities
    probs = model.predict(X_test, batch_size=32, verbose=0)
    preds = np.argmax(probs, axis=1)
    predictions_probs[name] = probs
    predictions_classes[name] = preds
    
    # Calculate classification metrics
    metrics = evaluate_classification_metrics(y_test, probs, class_names=class_names)
    
    # Measure inference latency
    speed_info = measure_inference_speed(model, X_test, batch_size=32)
    
    metrics["model"] = name
    metrics["parameters"] = params[name]
    metrics["latency_ms"] = speed_info["latency_per_sample_ms"]
    metrics["throughput_fps"] = speed_info["throughput_samples_per_sec"]
    
    results.append(metrics)

df_comparison = pd.DataFrame(results)
cols_order = ["model", "accuracy", "macro_f1", "macro_precision", "macro_recall", "roc_auc_macro", "latency_ms", "parameters"]
df_summary = df_comparison[cols_order].sort_values(by="macro_f1", ascending=False).reset_index(drop=True)
df_summary


---
### 4. Comparative Visualizations
We visualize Accuracy, Macro F1-Score, and Inference Latency side-by-side.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=df_summary, x="model", y="macro_f1", hue="model", legend=False, ax=axes[0], palette="viridis")
axes[0].set_title("Macro F1-Score Comparison")
axes[0].set_ylim([0, 1.0])
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height()), ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")

sns.barplot(data=df_summary, x="model", y="accuracy", hue="model", legend=False, ax=axes[1], palette="magma")
axes[1].set_title("Test Accuracy Comparison")
axes[1].set_ylim([0, 1.0])
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height()), ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")

sns.barplot(data=df_summary, x="model", y="latency_ms", hue="model", legend=False, ax=axes[2], palette="crest")
axes[2].set_title("Inference Latency (ms / sample)")
for p in axes[2].patches:
    axes[2].annotate(f"{p.get_height():.3f} ms", (p.get_x() + p.get_width() / 2., p.get_height()), ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")

plt.tight_layout()
plt.show()


---
### 5. Confusion Matrix Heatmaps (2x2 Grid)
We display the confusion matrix for each model.

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, preds) in enumerate(predictions_classes.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[idx])
    axes[idx].set_title(f"{name} - Confusion Matrix")
    axes[idx].set_xlabel("Predicted Label")
    axes[idx].set_ylabel("True Label")

plt.tight_layout()
plt.show()


---
### 6. Multi-Class ROC-AUC Curve Comparison
Overlaying Receiver Operating Characteristic curves across models.

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
fig, ax = plt.subplots(figsize=(9, 6))

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
for idx, (name, probs) in enumerate(predictions_probs.items()):
    # Macro-averaged ROC
    fpr = dict()
    tpr = dict()
    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs[:, i])
    
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(num_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= num_classes
    
    macro_auc = auc(all_fpr, mean_tpr)
    ax.plot(all_fpr, mean_tpr, color=colors[idx], lw=2, label=f"{name} (Macro AUC = {macro_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1.5)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Multi-Class Macro ROC Curves Comparison")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.show()


---
### 7. Overlay Learning Curves (Training & Validation History)
Comparing convergence speed, training stability, and validation loss across models.

In [ ]:
histories = {}
hist_paths = {
    "SimpleRNN": os.path.join(MODEL_DIR, "rnn_history.pkl"),
    "LSTM":      os.path.join(MODEL_DIR, "lstm_history.pkl"),
    "GRU":       os.path.join(MODEL_DIR, "gru_history.pkl"),
    "1D-CNN":    os.path.join(MODEL_DIR, "cnn_history.pkl")
}

for name, path in hist_paths.items():
    if os.path.exists(path):
        with open(path, "rb") as f:
            histories[name] = pickle.load(f)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for name, hist in histories.items():
    if "val_loss" in hist:
        axes[0].plot(hist["val_loss"], label=f"{name} Val Loss")
    if "val_accuracy" in hist:
        axes[1].plot(hist["val_accuracy"], label=f"{name} Val Acc")

axes[0].set_title("Validation Loss Comparison")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation Accuracy Comparison")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
### 8. Critical Analysis & Discussion

#### Key Empirical Findings:
1. **Predictive Performance**: Gated architectures (GRU and LSTM) and 1D-CNN outperform standard SimpleRNN on sequential text context. SimpleRNN suffers from vanishing gradients on 200-token sequences.
2. **Computational Efficiency**: 1D-CNN achieves the lowest latency per sample during inference due to parallel matrix convolutions, whereas recurrent architectures process tokens sequentially.
3. **Parameter Complexity**: GRU offers a parameter-efficient alternative to LSTM (having 3 gate mechanisms instead of 4), yielding comparable accuracy with lower training cost.
4. **Generalization & Overfitting**: Class weighting effectively mitigates target imbalance, while dropout (0.5) stabilizes validation accuracy curves across epochs.